In [3]:
# imports

import pandas as pd
import numpy as np
import json
import polars as pl
from pathlib import Path
# There is also auto tokenizer for general purpose but just using BERT for simplicity
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
import torch

c:\Users\aca23lr\source\dissertation\tabfact_bert_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, label, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, label in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [16]:
# Load in table dataframe
def read_in_entire_table_contents():
    return (
        pl.read_csv(
            "../tabfact_data/tables/*.csv",
            has_header=False,
            separator="\1", # Non-existent delimiter so full line/file stays intact
            new_columns=["table_content"],
            include_file_paths="file_name",
        )
        .group_by("file_name", maintain_order=True) # Group all the table lines together 
        .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
        .to_pandas()
    )

# Load in table dataframe
import polars as pl

def read_in_column_labelled_rows():
    entire_table_content = read_in_entire_table_contents()
    return entire_table_content




df_tables = read_in_column_labelled_rows()

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

df_tables["headers"] = df_tables["table_content"].apply(lambda x: x.split("\n")[0].split("#")) # Get the first line of the table content as headers

df_tables["rows"] = df_tables["table_content"].apply(lambda x: [row.split("#") for row in x.split("\n")[1:]]) # Get the rest of the lines as rows

df_tables["column_labelled_rows"] = df_tables.apply(
    lambda row: "\n".join(
        "ROW | " + ", ".join(f"{h}: {v}" for h, v in zip(row["headers"], r))
        for r in row["rows"]
    ),
    axis=1,
)

print(df_tables["column_labelled_rows"].iloc[0])

ROW | aircraft: robinson r - 22, description: light utility helicopter, max gross weight: 1370 lb (635 kg), total disk area: 497 ft square (46.2 m square), max disk loading: 2.6 lb / ft square (14 kg / m square)
ROW | aircraft: bell 206b3 jetranger, description: turboshaft utility helicopter, max gross weight: 3200 lb (1451 kg), total disk area: 872 ft square (81.1 m square), max disk loading: 3.7 lb / ft square (18 kg / m square)
ROW | aircraft: ch - 47d chinook, description: tandem rotor helicopter, max gross weight: 50000 lb (22680 kg), total disk area: 5655 ft square (526 m square), max disk loading: 8.8 lb / ft square (43 kg / m square)
ROW | aircraft: mil mi - 26, description: heavy - lift helicopter, max gross weight: 123500 lb (56000 kg), total disk area: 8495 ft square (789 m square), max disk loading: 14.5 lb / ft square (71 kg / m square)
ROW | aircraft: ch - 53e super stallion, description: heavy - lift helicopter, max gross weight: 73500 lb (33300 kg), total disk area: 490

In [4]:
# Load in non table data frames

df_full_cleaned = load_and_convert_data_to_dataframe("../tabfact_data/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['label'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['label'] == 1]

df_r1_training = load_and_convert_data_to_dataframe("../tabfact_data/r1_training_all.json")
df_r2_training = load_and_convert_data_to_dataframe("../tabfact_data/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe("../tabfact_data/train_examples.json")
df_test = load_and_convert_data_to_dataframe("../tabfact_data/test_examples.json")
df_validation = load_and_convert_data_to_dataframe("../tabfact_data/val_examples.json")


In [5]:
# Returns a pandas dataframe that contains a dataframe linking the table content with a statement and entailed status
def get_table_claim_df(df, whole_table=True, table_before_claim=True):
    if table_before_claim:
        df_table_claim = pd.merge(
            df_tables[["file_name", "table_content"]],
            df[["table_id", "statement", "label"]],
            left_on="file_name",
            right_on="table_id",
            how="inner" # Join matching pairs
        )
    else:
        df_table_claim = pd.merge(
            df[["table_id", "statement", "label"]],
            df_tables[["file_name", "table_content"]],
            left_on="table_id",
            right_on="file_name",
            how="inner" # Join matching pairs
        )

    if table_before_claim:
        df_table_claim["table_claim"] = "TABLE: " + df_table_claim["table_content"] + " | CLAIM: " + df_table_claim["statement"]
    else:
        df_table_claim["table_claim"] = "CLAIM: " + df_table_claim["statement"] + " | TABLE: " + df_table_claim["table_content"]

    return df_table_claim[["table_claim", "label"]]


# Get train, validation, and test data
df_train_table_claim = get_table_claim_df(df_training)
df_val_table_claim = get_table_claim_df(df_validation)
df_test_table_claim = get_table_claim_df(df_test)

print(f"train:\n {df_train_table_claim["label"].value_counts()}\n")
print(f"val:\n {df_val_table_claim["label"].value_counts()}\n")
print(f"test:\n {df_test_table_claim["label"].value_counts()}\n")

train:
 label
1    50820
0    41463
Name: count, dtype: int64

val:
 label
1    6478
0    6314
Name: count, dtype: int64

test:
 label
1    6425
0    6354
Name: count, dtype: int64



In [6]:
# Define BERT model and tokenizer
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 351.70it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [16]:
def tokenize_function(col):
    return bert_tokenizer(col['table_claim'], padding='max_length', truncation=True)

# Convert to dataset object for BERT training
dataset_train_table_claim = Dataset.from_pandas(df_train_table_claim)
dataset_val_table_claim = Dataset.from_pandas(df_val_table_claim)

datasets = DatasetDict({
    "train": dataset_train_table_claim,
    "val": dataset_val_table_claim
})

tokenized_datasets = datasets.map(tokenize_function, batched=True, remove_columns=["table_claim"])


Map: 100%|██████████| 12792/12792 [00:03<00:00, 3297.16 examples/s]


In [17]:
# tokenized_datasets["train"][:5]

# datasets["train"].features
tokenized_datasets["train"].features

{'label': Value('int64'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

In [ ]:
# Training the model

if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
    training_args = TrainingArguments(
    output_dir='./results',
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit="2",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy", 
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=True
    )

    trainer = Trainer(
        model=bert_model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['val']
    )

    trainer.train()
else:
    print("GPU NOT active. Running on CPU. Training will not commence")


c:\Users\aca23lr\source\dissertation\tabfact_bert_classification\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 